# Tech Challenge — Fine-Tuning completo do Qwen com MedQuAD

**Objetivo:** demonstrar a preparação de perguntas e respostas médicas, o treinamento experimental de um modelo de linguagem e a avaliação inicial de sua geração.

**Modelo:** Qwen2.5-0.5B-Instruct · **Dados:** MedQuAD processado · **Ambiente:** Google Colab / GPU · **Estratégia:** Fine-Tuning

## 1. Verificação do ambiente

Confirmamos a disponibilidade de CUDA e o modelo da GPU. A GPU acelera operações matriciais necessárias ao treinamento.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
CUDA disponível: True
GPU: Tesla T4


## 2. Instalação de dependências

Instalamos bibliotecas para modelos Transformers, datasets e treinamento.

In [ ]:
%pip install -q transformers datasets accelerate

## 3. Conferência das versões

Registramos as versões para facilitar a reprodução e o diagnóstico de incompatibilidades.

In [ ]:
import transformers
import datasets
import accelerate

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)

Transformers: 5.22.1
Datasets: 4.8.5
Accelerate: 1.15.0


## 4. Montagem do Google Drive

O armazenamento do Colab é temporário. Montar o Drive permite acessar os JSONL processados e salvar os pesos treinados.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 5. Carregamento do MedQuAD processado

Lemos `train.jsonl` e `validation.jsonl`. O conjunto de teste, separado anteriormente, não é utilizado nesta etapa. Os campos esperados são `instruction` (pergunta) e `response` (resposta).

In [ ]:
from datasets import load_dataset

base_path = "/content/drive/MyDrive/TechChallenge3/data/processed"

dataset = load_dataset(
    "json",
    data_files={
        "train": f"{base_path}/train.jsonl",
        "val": f"{base_path}/val.jsonl"
    }
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response'],
        num_rows: 13076
    })
    val: Dataset({
        features: ['instruction', 'response'],
        num_rows: 1645
    })
})


## 6. Inspeção de um registro

Verificamos se a estrutura dos dados corresponde ao esperado antes de iniciar o processamento.

In [ ]:
example = dataset["train"][0]

print("Pergunta:", example["instruction"])
print("Resposta:", example["response"][:500])

Pergunta: Is primary myelofibrosis inherited ?
Resposta: This condition is generally not inherited but arises from gene mutations that occur in early blood-forming cells after conception. These alterations are called somatic mutations.


## 7. Modelo base e tokenizer

Usamos `Qwen/Qwen2.5-0.5B-Instruct`. **Full Fine-Tuning** atualiza todos os pesos treináveis, ao contrário de LoRA. Carregamos os pesos em FP32 para compatibilidade com o escalonamento de gradientes no treinamento de precisão mista FP16.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Carrega o tokenizer, responsável por converter texto em tokens.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Carrega o modelo base em FP32: os pesos treináveis não ficam em FP16.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32,
    device_map="auto"
)

# Full Fine-Tuning: não congelamos nenhuma camada.
model.requires_grad_(True)

model.config.use_cache = False
model.gradient_checkpointing_enable()

print("Modelo carregado com sucesso!")
print("Dispositivo:", next(model.parameters()).device)
print("Precisão dos pesos:", next(model.parameters()).dtype)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Modelo carregado com sucesso!
Dispositivo: cuda:0
Precisão dos pesos: torch.float32


## 8. Conferência dos parâmetros treináveis

Somamos os parâmetros do modelo e verificamos que todos estão habilitados para treinamento.

In [ ]:
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

total_params = sum(
    p.numel() for p in model.parameters()
)

print(f"Parâmetros treináveis: {trainable_params:,}")
print(f"Parâmetros totais: {total_params:,}")
print(f"Percentual treinável: {100 * trainable_params / total_params:.2f}%")

Parâmetros treináveis: 494,032,768
Parâmetros totais: 494,032,768
Percentual treinável: 100.00%


## 9. Formatação no padrão de conversa

O *chat template* converte pergunta e resposta em uma sequência com papéis `user` e `assistant`, seguindo o formato esperado pelo Qwen.

In [ ]:
# Converte cada par pergunta/resposta no formato conversacional do Qwen.
def format_example(example):
    messages = [
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["response"]
        }
    ]

    # Não acrescenta um prompt de geração: a resposta já está presente.
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": formatted_text}

## 10. Aplicação da formatação

Aplicamos a função a todos os registros e criamos uma coluna `text` sem descartar os campos originais.

In [ ]:
formatted_dataset = dataset.map(format_example)

print(formatted_dataset)

Map:   0%|          | 0/13076 [00:00<?, ? examples/s]

Map:   0%|          | 0/1645 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response', 'text'],
        num_rows: 13076
    })
    val: Dataset({
        features: ['instruction', 'response', 'text'],
        num_rows: 1645
    })
})


## 11. Inspeção do texto formatado

Conferimos os marcadores especiais da conversa para garantir que a entrada segue o formato do modelo.

In [ ]:
print(formatted_dataset["train"][0]["text"][:1200])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Is primary myelofibrosis inherited ?<|im_end|>
<|im_start|>assistant
This condition is generally not inherited but arises from gene mutations that occur in early blood-forming cells after conception. These alterations are called somatic mutations.<|im_end|>



## 12. Subconjunto experimental

Selecionamos 200 exemplos de treinamento e 50 de validação para verificar o pipeline rapidamente. **Este teste não representa treinamento completo nos 13 mil exemplos.**

In [ ]:
train_sample = formatted_dataset["train"].select(range(200))
val_sample = formatted_dataset["val"].select(range(50))

print("Treinamento:", len(train_sample))
print("Validação:", len(val_sample))

Treinamento: 200
Validação: 50


## 13. Teste inicial de tokenização

O tokenizer converte texto em IDs numéricos. Inspecionamos o comprimento de um exemplo com limite de 512 tokens.

In [ ]:
example = train_sample[0]["text"]

tokens = tokenizer(
    example,
    truncation=True,
    max_length=512
)

print("Quantidade de tokens:", len(tokens["input_ids"]))
print("Primeiros 20 tokens:", tokens["input_ids"][:20])

Quantidade de tokens: 70
Primeiros 20 tokens: [151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645]


## 14. Distribuição dos comprimentos

Medimos quantos tokens cada exemplo ocupa para definir uma estratégia que evite truncar respostas médicas.

In [ ]:
import numpy as np

train_lengths = [
    len(tokenizer(example["text"])["input_ids"])
    for example in train_sample
]

print("Menor exemplo:", min(train_lengths))
print("Maior exemplo:", max(train_lengths))
print("Média:", round(np.mean(train_lengths), 2))
print("Mediana:", round(np.median(train_lengths), 2))
print("Percentil 90:", round(np.percentile(train_lengths, 90), 2))
print("Exemplos acima de 512:", sum(length > 512 for length in train_lengths))

Menor exemplo: 60
Maior exemplo: 2642
Média: 275.04
Mediana: 197.5
Percentil 90: 505.8
Exemplos acima de 512: 20


## 15. Economia de memória: gradient checkpointing

O *gradient checkpointing* recalcula parte das ativações durante a retropropagação para reduzir o uso de VRAM, à custa de processamento extra. Desabilitamos o cache de geração durante o treinamento.

In [ ]:
model.gradient_checkpointing_enable()

model.config.use_cache = False

print("Gradient checkpointing ativado!")
print("Use cache:", model.config.use_cache)

Gradient checkpointing ativado!
Use cache: False


## 16. Filtragem por comprimento

Excluímos do **experimento** os exemplos que excedem 512 tokens. Isso evita respostas cortadas, mas reduz a cobertura do conjunto e deve ser revisto num treinamento maior.

In [ ]:
# Limite conservador para o primeiro experimento; exemplos longos são excluídos.
MAX_LENGTH = 512

def fits_max_length(example):
    tokens = tokenizer(
        example["text"],
        add_special_tokens=False
    )

    return len(tokens["input_ids"]) <= MAX_LENGTH

train_filtered = train_sample.filter(fits_max_length)
val_filtered = val_sample.filter(fits_max_length)

print("Treinamento original:", len(train_sample))
print("Treinamento filtrado:", len(train_filtered))

print("Validação original:", len(val_sample))
print("Validação filtrada:", len(val_filtered))

Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50 [00:00<?, ? examples/s]

Treinamento original: 200
Treinamento filtrado: 180
Validação original: 50
Validação filtrada: 44


## 17. Construção dos labels e máscara de perda

A pergunta é contexto, não alvo de previsão. Marcamos seus tokens com `-100` para que não contribuam para a *loss*. Os tokens da resposta permanecem como alvos. Conferimos o alinhamento dos tokens.

In [ ]:
def tokenize_example(example):
    text = example["text"]

    # Marcador que identifica o início da resposta
    assistant_marker = "<|im_start|>assistant\n"

    if assistant_marker not in text:
        raise ValueError("Marcador do assistente não encontrado")

    # Separa o texto antes da resposta
    prefix, _ = text.split(assistant_marker, 1)
    prefix += assistant_marker

    # Tokeniza a conversa completa
    full_tokens = tokenizer(
        text,
        add_special_tokens=False
    )["input_ids"]

    # Tokeniza apenas o prefixo
    prefix_tokens = tokenizer(
        prefix,
        add_special_tokens=False
    )["input_ids"]

    # Confere se os tokens estão alinhados
    if full_tokens[:len(prefix_tokens)] != prefix_tokens:
        raise ValueError("Os tokens do prefixo não estão alinhados")

    prefix_length = len(prefix_tokens)

    # Os labels começam como cópia dos IDs da conversa inteira.
    labels = full_tokens.copy()

    # -100 indica ao cálculo da perda que a pergunta não deve ser prevista.
    labels[:prefix_length] = [-100] * prefix_length

    return {
        "input_ids": full_tokens,
        "attention_mask": [1] * len(full_tokens),
        "labels": labels
    }

## 18. Tokenização dos subconjuntos

Geramos `input_ids`, `attention_mask` e `labels` e removemos colunas de texto não necessárias ao `Trainer`.

In [ ]:
train_tokenized = train_filtered.map(
    tokenize_example,
    remove_columns=train_filtered.column_names
)

val_tokenized = val_filtered.map(
    tokenize_example,
    remove_columns=val_filtered.column_names
)

print(train_tokenized)
print(val_tokenized)

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 180
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 44
})


## 19. Inspeção da máscara

Contamos tokens ignorados e tokens usados na *loss* e decodificamos somente os alvos da resposta.

In [ ]:
sample = train_tokenized[0]

ignored_tokens = sum(
    label == -100 for label in sample["labels"]
)

trained_tokens = sum(
    label != -100 for label in sample["labels"]
)

print("Tokens ignorados:", ignored_tokens)
print("Tokens utilizados na loss:", trained_tokens)

print(
    "Resposta:",
    tokenizer.decode(
        [token for token in sample["labels"] if token != -100]
    )[:500]
)

Tokens ignorados: 38
Tokens utilizados na loss: 32
Resposta: This condition is generally not inherited but arises from gene mutations that occur in early blood-forming cells after conception. These alterations are called somatic mutations.<|im_end|>



## 20. Validação dos exemplos tokenizados

Verificamos comprimentos, limite máximo, presença de tokens supervisionados e correspondência entre labels e IDs.

In [ ]:
def validate_tokenized_dataset(data, max_length=512):
    for index, example in enumerate(data):
        input_ids = example["input_ids"]
        labels = example["labels"]
        attention_mask = example["attention_mask"]

        assert len(input_ids) == len(labels) == len(attention_mask), (
            f"Tamanhos incompatíveis para o exemplo {index}"
        )

        assert len(input_ids) <= max_length, (
            f"O exemplo {index} ultrapassou o limite"
        )

        assert any(label != -100 for label in labels), (
            f"O exemplo {index} não possui tokens de resposta"
        )

        assert all(
            label == -100 or label == token
            for label, token in zip(labels, input_ids)
        ), f"Labels incorretos no exemplo {index}"

    print(f"Validação concluída: {len(data)} exemplos corretos!")

validate_tokenized_dataset(train_tokenized)
validate_tokenized_dataset(val_tokenized)

Validação concluída: 180 exemplos corretos!
Validação concluída: 44 exemplos corretos!


## 21. Data collator: padding dinâmico

Cada lote pode reunir sequências de comprimentos diferentes. Adicionamos tokens de padding ao final; a máscara de atenção vale `0` nesses tokens, e os labels de padding recebem `-100`.

In [ ]:
import torch

def data_collator(features):
    # O maior exemplo define o comprimento do lote, evitando padding fixo.
    max_length = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_masks = []
    labels = []

    for feature in features:
        # O padding não é supervisionado (label=-100).
        padding_length = max_length - len(feature["input_ids"])

        input_ids.append(
            feature["input_ids"]
            + [tokenizer.pad_token_id] * padding_length
        )

        attention_masks.append(
            feature["attention_mask"]
            + [0] * padding_length
        )

        labels.append(
            feature["labels"]
            + [-100] * padding_length
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long)
    }

## 22. Teste do data collator

Comparamos as formas de `input_ids`, `attention_mask` e `labels`. Os três tensores devem ter dimensões idênticas.

In [ ]:
batch = data_collator([
    train_tokenized[0],
    train_tokenized[1]
])

print("Input IDs:", batch["input_ids"].shape)
print("Attention mask:", batch["attention_mask"].shape)
print("Labels:", batch["labels"].shape)

Input IDs: torch.Size([2, 382])
Attention mask: torch.Size([2, 382])
Labels: torch.Size([2, 382])


## 23. Hiperparâmetros do treinamento

Usamos *batch size* 1 e acumulação de 8 passos: aproximadamente oito micro-lotes por atualização dos pesos. Dez passos são um **teste de funcionamento**, não uma otimização definitiva.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen-medquad-full-ft",

    # Configuração do treinamento
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,  # 8 micro-lotes por atualização (batch efetivo ~8)
    max_steps=10,  # Ensaio rápido: não é treinamento completo

    # Avaliação e acompanhamento
    eval_strategy="steps",
    eval_steps=5,
    logging_steps=1,
    save_strategy="no",

    # Otimização de memória
    gradient_checkpointing=True,
    fp16=True,  # Operações em precisão mista; pesos carregados em FP32
    optim="adamw_torch",

    # Outras configurações
    report_to="none",
    remove_unused_columns=False
)

print("Configuração criada com sucesso!")

Configuração criada com sucesso!


## 24. Inicialização do Trainer

O `Trainer` conecta modelo, hiperparâmetros, conjuntos de dados e função de montagem de lotes.

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator
)

print("Trainer configurado com sucesso!")

Trainer configurado com sucesso!


## 25. Treinamento experimental e métricas

Executamos dez atualizações. `train_loss` resume a perda de treinamento; `eval_loss` mede a perda no conjunto de validação. Queda na perda **não comprova precisão médica**.

In [ ]:
# Executa o treino e as avaliações periódicas definidas em TrainingArguments.
train_result = trainer.train()

print("Treinamento experimental concluído!")
print(train_result.metrics)

Step,Training Loss,Validation Loss
5,1.893983,1.858097
10,1.196313,1.686361


Treinamento experimental concluído!
{'train_runtime': 24.0151, 'train_samples_per_second': 3.331, 'train_steps_per_second': 0.416, 'total_flos': 37710292468992.0, 'train_loss': 1.7550650477409362, 'epoch': 0.4444444444444444}


### Interpretação dos resultados observados

Na execução experimental anterior, a perda de treinamento caiu de **1,894** (passo 5) para **1,197** (passo 10), enquanto a perda de validação caiu de **1,858** para **1,686**. Isso sugere aprendizado no subconjunto utilizado, mas a amostra e a duração são pequenas. Não é possível concluir que o modelo generaliza bem ou que produz respostas clinicamente corretas.

**Atenção:** o conjunto de teste separado não foi avaliado neste notebook. A comparação com o modelo base também permanece pendente.

## 26. Salvamento do modelo ajustado

Salvamos os pesos completos e o tokenizer no Google Drive. `save_strategy="no"` desativa checkpoints automáticos; por isso o salvamento manual é indispensável.

In [ ]:
import os

save_path = "/content/drive/MyDrive/TechChallenge3/models/qwen-medquad-full-ft-test"

os.makedirs(save_path, exist_ok=True)

# Full Fine-Tuning: salva os pesos completos, não apenas adaptadores.
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Modelo salvo em:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo salvo em: /content/drive/MyDrive/TechChallenge3/models/qwen-medquad-full-ft-test


## 27. Inferência com o modelo ajustado

Colocamos o modelo em modo de avaliação, formatamos uma pergunta sem resposta e geramos texto. `torch.no_grad()` evita calcular gradientes.

In [ ]:
# Inferência: desativa o comportamento de treinamento (ex.: dropout).
model.eval()

question = "Is primary myelofibrosis inherited?"

messages = [
    {"role": "user", "content": question}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True  # Abre a posição da resposta do assistente
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

In [ ]:
import torch

# Inferência não requer gradientes e usa menos memória.
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,  # Geração determinística para comparação
        pad_token_id=tokenizer.pad_token_id
    )

# Remove os tokens da pergunta, deixando apenas os tokens gerados.
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("Pergunta:", question)
print("Resposta:", answer)

Pergunta: Is primary myelofibrosis inherited?
Resposta: Primary myelofibrosis is an inherited condition that occurs in one in 10,000 people. It usually affects men and women of the same sex. The condition is caused by mutations in the MYH7 gene. This gene provides instructions for making a protein called myoglobin, which helps cells survive and grow. Myoglobin also helps remove waste products from cells. In primary myelofibrosis, abnormal myoglobin accumulates in the blood and causes fibrous tissue to form in various parts of the body. These changes can cause the bones to become fragile and brittle (bone fragility), leading to fractures. Primary myelofibrosis is not contagious.


## 28. Discussão crítica e próximos passos

**Exemplo de limitação observado:** ao responder à pergunta *“Is primary myelofibrosis inherited?”*, o modelo gerou afirmações incorretas, incluindo que a doença seria geralmente hereditária e que estaria associada ao gene `MYH7`. A mielofibrose primária geralmente está associada a mutações somáticas adquiridas; uma resposta fluente não é garantia de exatidão.

**Conclusões do experimento:**

1. O pipeline de Full Fine-Tuning foi executado com sucesso em um subconjunto de dados.
2. A redução da *loss* é um indicador de otimização, não de segurança clínica.
3. Precisamos avaliar um conjunto de teste isolado, comparar com o modelo base e examinar factualidade e fontes.
4. O próximo módulo do projeto utilizará **RAG** com protocolos hospitalares fictícios, rastreabilidade de fontes, controle de acesso simulado e revisão humana. RAG reduz dependência exclusiva da memória paramétrica, mas **não elimina automaticamente alucinações**.

**Para a apresentação:** mostrar a resposta incorreta como evidência de uma limitação real e explicar por que o assistente não pode atuar de maneira autônoma em decisões médicas.

29. Preparação do conjunto de teste
Nesta etapa, carregamos o conjunto de teste do MedQuAD, separado anteriormente para avaliar o modelo com perguntas não utilizadas no treinamento ou na validação.

Nosso objetivo é comparar as respostas do modelo original e do modelo ajustado, observando a precisão factual, a relevância e possíveis alucinações.

In [ ]:
from datasets import load_dataset
import os

# Caminho do conjunto de teste reservado na preparação do MedQuAD.
test_path = (
    "/content/drive/MyDrive/TechChallenge3/"
    "data/processed/test.jsonl"
)

# Verifica se o arquivo está disponível antes de carregá-lo.
if not os.path.isfile(test_path):
    raise FileNotFoundError(
        f"Arquivo de teste não encontrado: {test_path}"
    )

# Carrega os registros sem modificar os dados originais.
test_dataset = load_dataset(
    "json",
    data_files={"test": test_path}
)["test"]

print("Total de exemplos de teste:", len(test_dataset))

# Inspeciona um exemplo para conferir sua estrutura.
print("\nPergunta:", test_dataset[0]["instruction"])
print("Resposta de referência:", test_dataset[0]["response"][:500])

Generating test split: 0 examples [00:00, ? examples/s]

Total de exemplos de teste: 1638

Pergunta: What is (are) Familial Mediterranean fever ?
Resposta de referência: Familial Mediterranean fever (FMF) is an inherited condition characterized by episodes of painful inflammation of the abdominal lining (peritonitis), lining surrounding the lungs (pleurisy), and joints (arthralgia and occasionally arthritis). These episodes are often accompanied by fever and sometimes a characteristic ankle rash. The first episode usually occurs in childhood or the teenage years, but in some cases, the initial attack occurs much later in life. Between attacks, people often do no


30. Seleção de perguntas para avaliação comparativa
Para uma avaliação exploratória, selecionamos cinco exemplos do conjunto de teste reservado.

Cada pergunta será enviada ao modelo original e ao modelo ajustado, mantendo as mesmas configurações de geração.

As respostas serão comparadas com as referências do MedQuAD. Essa avaliação qualitativa não substitui uma análise sistemática de precisão clínica.

In [ ]:
# Seleciona cinco exemplos do conjunto de teste.
# A seleção é fixa para permitir reproduzir a comparação.
test_samples = test_dataset.select(range(5))

# Exibe as perguntas que serão utilizadas.
for index, example in enumerate(test_samples):
    print(f"\nPergunta {index + 1}:")
    print(example["instruction"])


Pergunta 1:
What is (are) Familial Mediterranean fever ?

Pergunta 2:
What is (are) familial Mediterranean fever ?

Pergunta 3:
Is congenital hypothyroidism inherited ?

Pergunta 4:
What is (are) adermatoglyphia ?

Pergunta 5:
What are the symptoms of Werner's syndrome ?


31. Função de inferência
Nesta etapa, criamos uma função reutilizável para gerar respostas a partir de um modelo de linguagem.

A função aplica o formato de conversa esperado pelo Qwen, realiza a tokenização e executa a geração sem calcular gradientes.

Utilizamos geração determinística para facilitar a comparação dos resultados.

In [ ]:
import torch

def generate_answer(model, tokenizer, question, max_new_tokens=200):
    # Coloca o modelo em modo de avaliação.
    model.eval()

    # Constrói a conversa no formato esperado pelo Qwen.
    messages = [
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Converte o texto em tokens e envia ao dispositivo do modelo.
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(next(model.parameters()).device)

    # Gera a resposta sem calcular gradientes.
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    # Remove os tokens correspondentes à pergunta.
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    # Converte os tokens gerados em texto.
    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

32. Inferência com o modelo ajustado
Executamos o Qwen após o Full Fine-Tuning utilizando cinco perguntas do conjunto de teste.

As respostas serão armazenadas para comparação posterior com o modelo original.

Nesta etapa, não calculamos gradientes nem atualizamos os pesos.

In [ ]:
# Armazena as respostas do modelo ajustado.
finetuned_answers = []

for index, example in enumerate(test_samples):
    question = example["instruction"]

    # Gera uma resposta utilizando o modelo treinado.
    answer = generate_answer(
        model,
        tokenizer,
        question
    )

    finetuned_answers.append(answer)

    print(f"\n{'=' * 60}")
    print(f"PERGUNTA {index + 1}: {question}")
    print(f"\nRESPOSTA DO MODELO AJUSTADO:\n{answer}")


PERGUNTA 1: What is (are) Familial Mediterranean fever ?

RESPOSTA DO MODELO AJUSTADO:
Familial Mediterranean fever is an inherited disorder that affects the skin and mucosa of the face and neck. It occurs in one in 10,000 people. The condition usually begins during childhood or adolescence. It may be associated with other conditions such as: - Sjogren's syndrome - Multiple exudative cutaneous retinopathy - Pemphigus vulgaris - Raynaud's phenomenon - Stevens-Johnson syndrome - Thrombocytopenic purpura - Urticaria - Vasculitis - Wilm's tumor - X-linked recessive retinal atrophy - X-linked recessive retinal degeneration - X-linked recessive retinal dystrophy - X-linked recessive retinal pigmentary disorder - X-linked recessive retinal pigmentary disorder type 2A - X-linked recessive retinal pigmentary disorder type 2B - X-linked recessive retinal pigmentary disorder type 2C - X-linked recess

PERGUNTA 2: What is (are) familial Mediterranean fever ?

RESPOSTA DO MODELO AJUSTADO:
Familial

33. Registro das respostas do modelo ajustado
As respostas geradas pelo modelo após o Fine-Tuning serão armazenadas em um arquivo JSON.

Esse registro permitirá reproduzir a comparação, analisar erros e documentar as limitações do modelo sem repetir a inferência.

In [ ]:
import json
import os

results_path = (
    "/content/drive/MyDrive/TechChallenge3/"
    "results"
)

os.makedirs(results_path, exist_ok=True)

# Combina perguntas, referências e respostas geradas.
finetuned_results = []

for example, answer in zip(test_samples, finetuned_answers):
    finetuned_results.append({
        "question": example["instruction"],
        "reference": example["response"],
        "finetuned_answer": answer
    })

# Preserva caracteres Unicode e formata o arquivo.
output_file = os.path.join(
    results_path,
    "finetuned_evaluation.json"
)

with open(output_file, "w", encoding="utf-8") as file:
    json.dump(
        finetuned_results,
        file,
        ensure_ascii=False,
        indent=2
    )

print("Resultados salvos em:", output_file)
print("Total de respostas:", len(finetuned_results))

Resultados salvos em: /content/drive/MyDrive/TechChallenge3/results/finetuned_evaluation.json
Total de respostas: 5


34. Liberação da memória da GPU
Para comparar o modelo ajustado com o original, executaremos os modelos sequencialmente.

Isso reduz o consumo de memória da Tesla T4, evitando manter duas cópias do Qwen carregadas simultaneamente.

In [ ]:
import gc
import torch
import os

# Confirma que os pesos ajustados estão disponíveis.
model_path = (
    "/content/drive/MyDrive/TechChallenge3/"
    "models/qwen-medquad-full-ft-test"
)

assert os.path.isdir(model_path), (
    "Modelo ajustado não encontrado. Salve-o antes de continuar."
)

assert any(
    name.endswith(".safetensors") or name.endswith(".bin")
    for name in os.listdir(model_path)
), "Nenhum arquivo de pesos foi encontrado."

# Remove as referências ao modelo e ao Trainer.
del trainer
del model

# Solicita a coleta de objetos sem referências.
gc.collect()

# Libera blocos de memória CUDA não utilizados.
torch.cuda.empty_cache()

print("Modelo ajustado descarregado.")

Modelo ajustado descarregado.


35. Carregamento do modelo base
Carregamos o Qwen2.5-0.5B-Instruct original, sem os ajustes realizados com o MedQuAD.

Esse modelo será utilizado como referência para verificar se o Fine-Tuning produziu diferenças relevantes nas respostas.

Utilizamos FP16 porque não será necessário atualizar os pesos.

In [ ]:
from transformers import AutoModelForCausalLM
import torch

# Identificador do modelo original no Hugging Face.
base_model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Carrega o modelo sem os pesos do nosso Fine-Tuning.
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.float16,
    device_map="auto"
)

# Coloca o modelo em modo de inferência.
base_model.eval()

print("Modelo original carregado!")
print("Dispositivo:", next(base_model.parameters()).device)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Modelo original carregado!
Dispositivo: cuda:0


36. Avaliação do modelo original
Enviamos ao modelo original as mesmas cinco perguntas utilizadas anteriormente.

Mantemos a função de inferência e os parâmetros de geração para permitir uma comparação controlada.

In [ ]:
# Armazena as respostas do modelo original.
base_answers = []

for index, example in enumerate(test_samples):
    question = example["instruction"]

    # Reutiliza a função de inferência anterior.
    answer = generate_answer(
        base_model,
        tokenizer,
        question,
        max_new_tokens=200
    )

    base_answers.append(answer)

    print(f"\n{'=' * 60}")
    print(f"PERGUNTA {index + 1}: {question}")
    print(f"\nRESPOSTA DO MODELO ORIGINAL:\n{answer}")


PERGUNTA 1: What is (are) Familial Mediterranean fever ?

RESPOSTA DO MODELO ORIGINAL:
Familial Mediterranean fever (FMF) is an autoimmune disorder characterized by the presence of autoantibodies against the T-cell receptor (TCR) and/or the CD3 molecule on B cells. The disease is named after its association with the Mediterranean diet, which has been linked to increased risk of FMF in some families.

The exact cause of FMF is not fully understood, but it appears to be related to genetic factors and environmental exposures. Some studies suggest that certain dietary components, such as omega-3 fatty acids, may play a role in the development of FMF.

Symptoms of familial Mediterranean fever can include fatigue, joint pain, and skin rashes. Treatment typically involves medications to reduce inflammation and manage symptoms. In some cases, treatment may involve bone marrow transplantation or other medical interventions.

While there is no cure for familial Mediterranean fever, managing sym

39. Preparação do Full Fine-Tuning ampliado
Neste segundo experimento, utilizaremos todos os registros de treinamento que respeitem o limite de 512 tokens.

O conjunto de validação será utilizado para acompanhar a generalização do modelo. O conjunto de teste permanecerá reservado para avaliação final.

O treinamento começará com os pesos originais do Qwen2.5-0.5B-Instruct, sem reutilizar os pesos ajustados no experimento anterior.

In [ ]:
# Recupera todos os exemplos formatados.
full_train = formatted_dataset["train"]
full_val = formatted_dataset["val"]

# Aplica o mesmo filtro de comprimento do experimento anterior.
# Exemplos acima de 512 tokens são excluídos, não truncados.
full_train_filtered = full_train.filter(fits_max_length)
full_val_filtered = full_val.filter(fits_max_length)

print("Treinamento disponível:", len(full_train))
print("Treinamento elegível:", len(full_train_filtered))
print("Validação disponível:", len(full_val))
print("Validação elegível:", len(full_val_filtered))

# Tokeniza os conjuntos elegíveis, preservando a máscara
# que impede o cálculo da loss sobre a pergunta.
full_train_tokenized = full_train_filtered.map(
    tokenize_example,
    remove_columns=full_train_filtered.column_names
)

full_val_tokenized = full_val_filtered.map(
    tokenize_example,
    remove_columns=full_val_filtered.column_names
)

# Verifica se a tokenização produziu exemplos válidos.
validate_tokenized_dataset(full_train_tokenized)
validate_tokenized_dataset(full_val_tokenized)

print("\nDados preparados para o treinamento ampliado!")

Filter:   0%|          | 0/13076 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1645 [00:00<?, ? examples/s]

Treinamento disponível: 13076
Treinamento elegível: 11391
Validação disponível: 1645
Validação elegível: 1439


Map:   0%|          | 0/11391 [00:00<?, ? examples/s]

Map:   0%|          | 0/1439 [00:00<?, ? examples/s]

Validação concluída: 11391 exemplos corretos!
Validação concluída: 1439 exemplos corretos!

Dados preparados para o treinamento ampliado!


40. Inicialização do modelo para Full Fine-Tuning
Liberamos modelos anteriormente carregados na GPU e inicializamos novamente o Qwen2.5-0.5B-Instruct com seus pesos originais.

Mantemos os parâmetros treináveis, utilizamos pesos em FP32 e habilitamos gradient checkpointing para reduzir o consumo de memória durante o treinamento.

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM

# Remove referências de modelos e treinadores anteriores.
for name in ["trainer", "model", "base_model"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# Confirma que o Colab continua com GPU disponível.
assert torch.cuda.is_available(), (
    "GPU indisponível. Verifique o ambiente do Colab."
)

print("GPU:", torch.cuda.get_device_name(0))

# Carrega os pesos originais em FP32.
# O FP16 será usado pelo Trainer em precisão mista.
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    dtype=torch.float32,
    device_map="auto"
)

# Full Fine-Tuning: todos os parâmetros serão ajustados.
model.requires_grad_(True)

# Reduz o consumo de memória durante o treinamento.
model.config.use_cache = False
model.gradient_checkpointing_enable()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Parâmetros totais:", total_params)
print("Parâmetros treináveis:", trainable_params)
print("Tipo dos pesos:", next(model.parameters()).dtype)
print("Dispositivo:", next(model.parameters()).device)

GPU: Tesla T4


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Parâmetros totais: 494032768
Parâmetros treináveis: 494032768
Tipo dos pesos: torch.float32
Dispositivo: cuda:0


Verificar o armazenamento temporário local do Google Colab

In [ ]:
import shutil

total, used, free = shutil.disk_usage("/content")
gb = 1024 ** 3

print(f"Espaço total: {total / gb:.2f} GB")
print(f"Espaço utilizado: {used / gb:.2f} GB")
print(f"Espaço livre: {free / gb:.2f} GB")

Espaço total: 112.64 GB
Espaço utilizado: 3.10 GB
Espaço livre: 109.52 GB


41. Configuração do treinamento ampliado
O segundo experimento utilizará todos os 11.391 exemplos elegíveis, durante uma época.

Mantemos o otimizador AdamW, a taxa de aprendizado de 2e-5, a acumulação de oito gradientes e a precisão mista FP16, conforme a configuração que funcionou no experimento inicial.

Para acompanhar o treinamento, registraremos as perdas e salvaremos checkpoints periódicos contendo os estados necessários para sua retomada.

In [ ]:
import os
from pathlib import Path
from transformers import TrainingArguments, Trainer

# Diretório exclusivo do segundo experimento.
checkpoint_path = (
    "/content/checkpoints/qwen-medquad-full-ft-v2"
)

# Evita iniciar acidentalmente outro treinamento sobre
# checkpoints já existentes.
existing_checkpoints = list(
    Path(checkpoint_path).glob("checkpoint-*")
)

if existing_checkpoints:
    raise RuntimeError(
        "Já existem checkpoints neste diretório. "
        "Verifique a retomada antes de iniciar outro treino."
    )

training_args_v2 = TrainingArguments(
    output_dir=checkpoint_path,

    # Treinamento completo: uma passagem pelos dados.
    num_train_epochs=1,
    max_steps=-1,

    learning_rate=2e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Acompanhamento das métricas.
    logging_strategy="steps",
    logging_steps=25,

    # Avaliação ao longo da época.
    eval_strategy="steps",
    eval_steps=700,

    # Checkpoints para recuperar o progresso.
    save_strategy="steps",
    save_steps=700,
    save_total_limit=1,
    save_only_model=False,

    # Otimização de memória.
    gradient_checkpointing=True,
    fp16=True,
    optim="adamw_torch",

    # Reprodutibilidade e configuração do Trainer.
    seed=42,
    data_seed=42,
    report_to="none",
    remove_unused_columns=False
)

trainer_v2 = Trainer(
    model=model,
    args=training_args_v2,
    train_dataset=full_train_tokenized,
    eval_dataset=full_val_tokenized,
    data_collator=data_collator
)

print("Trainer do segundo experimento configurado.")
print("Exemplos de treino:", len(full_train_tokenized))
print("Exemplos de validação:", len(full_val_tokenized))
print("Épocas:", training_args_v2.num_train_epochs)
print("Checkpoint:", checkpoint_path)

Trainer do segundo experimento configurado.
Exemplos de treino: 11391
Exemplos de validação: 1439
Épocas: 1
Checkpoint: /content/checkpoints/qwen-medquad-full-ft-v2


42. Execução do Full Fine-Tuning ampliado
Executamos uma época de treinamento com os dados elegíveis do MedQuAD.

Acompanhamos a loss de treinamento e de validação para investigar o aprendizado e possíveis sinais de sobreajuste.

Ao final, salvamos o modelo completo e registramos as métricas observadas. A melhoria da precisão médica será investigada posteriormente por meio de avaliação comparativa.

In [ ]:
import json
import os

# Inicia o treinamento ampliado.
train_result_v2 = trainer_v2.train()

print("\nTreinamento ampliado concluído!")
print("Métricas:", train_result_v2.metrics)

# Diretório definitivo dos pesos ajustados.
final_model_path = (
    "/content/models/qwen-medquad-full-ft-v2"
)

os.makedirs(final_model_path, exist_ok=True)

# Salva os pesos completos e o tokenizer.
trainer_v2.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

# Recupera a última avaliação registrada pelo Trainer.
eval_logs = [
    entry for entry in trainer_v2.state.log_history
    if "eval_loss" in entry
]

last_evaluation = eval_logs[-1] if eval_logs else None

# Registra os resultados para o relatório e o vídeo.
metrics = {
    "training": train_result_v2.metrics,
    "last_evaluation": last_evaluation,
    "training_examples": len(full_train_tokenized),
    "validation_examples": len(full_val_tokenized),
    "epochs_requested": 1
}

metrics_file = os.path.join(
    final_model_path,
    "training_metrics.json"
)

with open(metrics_file, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2, default=str)

print("\nModelo salvo em:", final_model_path)
print("Última avaliação:", last_evaluation)
print("Métricas salvas em:", metrics_file)

Step,Training Loss,Validation Loss
700,1.141652,1.295597
1400,1.159428,1.216180
1424,1.159428,1.215799


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Treinamento ampliado concluído!
Métricas: {'train_runtime': 3735.3176, 'train_samples_per_second': 3.05, 'train_steps_per_second': 0.381, 'total_flos': 5497177567444992.0, 'train_loss': 1.2814904746044888, 'epoch': 1.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Modelo salvo em: /content/models/qwen-medquad-full-ft-v2
Última avaliação: {'eval_loss': 1.2157994508743286, 'eval_runtime': 66.834, 'eval_samples_per_second': 21.531, 'eval_steps_per_second': 21.531, 'epoch': 1.0, 'step': 1424}
Métricas salvas em: /content/models/qwen-medquad-full-ft-v2/training_metrics.json


43. Exportação e preservação do modelo ajustado
Após concluir o Full Fine-Tuning ampliado, verificamos se os arquivos do modelo foram salvos corretamente.

Em seguida, criamos um arquivo ZIP contendo os pesos ajustados, o tokenizer, a configuração e as métricas.

Esse arquivo será transferido para o computador local, evitando a perda do treinamento quando a sessão temporária do Google Colab for encerrada.

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

# Diretório onde salvamos o modelo treinado.
model_dir = Path(
    "/content/models/qwen-medquad-full-ft-v2"
)

# Verifica os arquivos essenciais.
assert model_dir.is_dir(), "Diretório do modelo não encontrado."
assert (model_dir / "config.json").exists(), (
    "Configuração do modelo não encontrada."
)

weight_files = list(model_dir.glob("*.safetensors"))

assert weight_files, "Pesos do modelo não encontrados."

# Mostra os arquivos presentes e seus tamanhos.
print("Arquivos do modelo:")
for file in sorted(model_dir.iterdir()):
    if file.is_file():
        size_mb = file.stat().st_size / (1024 ** 2)
        print(f"- {file.name}: {size_mb:.2f} MB")

# Cria ZIP sem recomprimir os pesos.
# Isso evita processamento desnecessário.
zip_path = Path(
    "/content/qwen-medquad-full-ft-v2.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_STORED,
    allowZip64=True
) as archive:

    for file in model_dir.rglob("*"):
        if file.is_file():
            archive.write(
                file,
                arcname=file.relative_to(model_dir.parent)
            )

print("\nArquivo ZIP criado!")
print(
    "Tamanho:",
    round(zip_path.stat().st_size / (1024 ** 3), 2),
    "GB"
)

# Solicita o download para seu computador.
files.download(str(zip_path))

Arquivos do modelo:
- chat_template.jinja: 0.00 MB
- config.json: 0.00 MB
- generation_config.json: 0.00 MB
- model.safetensors: 1884.62 MB
- tokenizer.json: 10.89 MB
- tokenizer_config.json: 0.00 MB
- training_args.bin: 0.00 MB
- training_metrics.json: 0.00 MB

Arquivo ZIP criado!
Tamanho: 1.85 GB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

44. Recuperação do ambiente após reinicialização
Nesta etapa, verificamos se os arquivos do modelo ajustado e os datasets continuam disponíveis.

Como o Google Colab utiliza armazenamento temporário, pode ser necessário restaurar os pesos do modelo a partir de uma cópia de segurança.

In [ ]:
from pathlib import Path
import torch

# Caminho do modelo que treinamos ontem.
model_path = Path(
    "/content/models/qwen-medquad-full-ft-v2"
)

# Caminho do arquivo ZIP gerado anteriormente.
zip_path = Path(
    "/content/qwen-medquad-full-ft-v2.zip"
)

# Caminho do dataset de teste no Google Drive.
test_path = Path(
    "/content/drive/MyDrive/TechChallenge3/"
    "data/processed/test.jsonl"
)

print("GPU disponível:", torch.cuda.is_available())
print("Modelo salvo no Colab:", model_path.exists())
print("ZIP salvo no Colab:", zip_path.exists())
print("Dataset de teste disponível:", test_path.exists())

GPU disponível: True
Modelo salvo no Colab: False
ZIP salvo no Colab: False
Dataset de teste disponível: True


## 44. Preparação do ambiente do assistente médico

Nesta etapa, preparamos as bibliotecas necessárias para implementar
o agente RAG, o banco vetorial ChromaDB e a orquestração LangGraph.

Utilizaremos o Google Colab em CPU, sem repetir o treinamento
do Qwen realizado anteriormente.

O Google Drive armazenará os dados sintéticos e os resultados
do projeto.

In [1]:
import sys
import subprocess
import importlib.util
from pathlib import Path

# Instala somente bibliotecas que ainda não estão disponíveis.
dependencias = {
    "langchain_chroma": "langchain-chroma",
    "langchain_huggingface": "langchain-huggingface",
    "langchain_core": "langchain-core",
    "langgraph": "langgraph",
    "sentence_transformers": "sentence-transformers",
    "transformers": "transformers"
}

faltando = [
    pacote
    for modulo, pacote in dependencias.items()
    if importlib.util.find_spec(modulo) is None
]

if faltando:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--quiet", *faltando
    ])

# Monta o Google Drive.
from google.colab import drive

drive.mount("/content/drive")

# Define a pasta principal do projeto.
BASE_PATH = Path(
    "/content/drive/MyDrive/TechChallenge3"
)

DATA_PATH = BASE_PATH / "data" / "synthetic"
RESULTS_PATH = BASE_PATH / "results"

DATA_PATH.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print("[OK] Ambiente preparado.")
print("Diretório do projeto:", BASE_PATH)

Mounted at /content/drive
[OK] Ambiente preparado.
Diretório do projeto: /content/drive/MyDrive/TechChallenge3


## 45. Criação da base de pacientes sintéticos

Criamos 1.000 prontuários inteiramente fictícios para demonstrar
a recuperação de informações hospitalares.

Cada paciente possui dados cadastrais, observações clínicas
e exames com status simulados.

Utilizamos uma semente fixa para permitir a reprodução dos dados.

Os registros são exclusivamente acadêmicos e não representam
pacientes reais.

In [2]:
import json
import random

# Arquivo persistente da base sintética.
PATIENTS_FILE = DATA_PATH / "patients_vector_v1.json"

if PATIENTS_FILE.exists():
    # Reutiliza a base existente sem sobrescrever alterações.
    with open(PATIENTS_FILE, "r", encoding="utf-8") as file:
        pacientes = json.load(file)

    print("Base existente carregada.")

else:
    rng = random.Random(42)

    primeiros_nomes = [
        "Marina", "Carlos", "Ana", "Pedro", "Juliana",
        "Rafael", "Fernanda", "Lucas", "Camila", "Bruno",
        "Beatriz", "Gabriel", "Larissa", "Eduardo", "Renata"
    ]

    sobrenomes = [
        "Almeida", "Costa", "Silva", "Oliveira",
        "Santos", "Souza", "Ferreira", "Pereira",
        "Rodrigues", "Martins", "Carvalho", "Lima"
    ]

    setores = [
        "Clínica Médica", "Observação",
        "Ambulatório", "Ortopedia"
    ]

    observacoes = [
        "Paciente relata episódios de cefaleia e tontura.",
        "Paciente relata desconforto abdominal intermitente.",
        "Paciente relata cansaço e dificuldade para dormir.",
        "Paciente apresenta relato de dor muscular.",
        "Paciente compareceu para acompanhamento de rotina."
    ]

    pacientes = []

    for numero in range(1, 1001):
        patient_id = f"PAC-DEMO-{numero:03d}"

        nome = (
            f"{rng.choice(primeiros_nomes)} "
            f"{rng.choice(sobrenomes)} "
            f"{rng.choice(sobrenomes)}"
        )

        # Preserva o paciente que utilizamos nos testes anteriores.
        if numero == 1:
            nome = "Marina Almeida Costa"

        observacao = (
            "Paciente relata episódios de cefaleia e tontura."
            if numero == 1
            else rng.choice(observacoes)
        )

        paciente = {
            "patient_id": patient_id,
            "nome": nome,
            "idade": rng.randint(18, 85),
            "email": f"paciente.{numero:04d}@hospital.invalid",
            "telefone": f"(00) 90000-{numero:04d}",
            "setor": rng.choice(setores),
            "observacoes": [
                {
                    "id": f"NOTA-{numero:04d}",
                    "texto": observacao
                }
            ],
            "exames": [
                {
                    "id": f"EX-{numero:04d}-01",
                    "nome": "Hemograma",
                    "status": (
                        "PENDENTE"
                        if numero % 3 != 0
                        else "CONCLUÍDO"
                    )
                },
                {
                    "id": f"EX-{numero:04d}-02",
                    "nome": "Glicemia",
                    "status": "CONCLUÍDO"
                }
            ]
        }

        pacientes.append(paciente)

    # Salva a base para reutilização entre sessões.
    with open(PATIENTS_FILE, "w", encoding="utf-8") as file:
        json.dump(
            pacientes,
            file,
            ensure_ascii=False,
            indent=2
        )

    print("Nova base sintética criada.")

# Verificações antes de continuar.
assert len(pacientes) == 1000
assert len({
    paciente["patient_id"] for paciente in pacientes
}) == 1000

print("Pacientes:", len(pacientes))
print("Arquivo:", PATIENTS_FILE)
print("Primeiro paciente:", pacientes[0]["nome"])
print("Último identificador:", pacientes[-1]["patient_id"])

Base existente carregada.
Pacientes: 1000
Arquivo: /content/drive/MyDrive/TechChallenge3/data/synthetic/patients_vector_v1.json
Primeiro paciente: Marina Almeida Costa
Último identificador: PAC-DEMO-1000


## 46. Indexação dos prontuários no ChromaDB

Nesta etapa, convertemos os dados sintéticos dos pacientes em
documentos que serão armazenados em um banco vetorial.

Cada paciente possui três documentos: cadastro, observações
clínicas e exames.

Utilizamos embeddings multilíngues para representar
semanticamente os documentos.

O índice é armazenado no Google Drive e poderá ser
reutilizado sem gerar os embeddings novamente.

O controle de acesso será implementado na próxima etapa.

In [3]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Diretório persistente para o banco vetorial.
CHROMA_PATH = BASE_PATH / "data" / "chroma_pacientes_v1"

# Modelo de embeddings multilíngue.
# Não é o Qwen; sua função é converter textos em vetores.
embeddings = HuggingFaceEmbeddings(
    model_name=(
        "sentence-transformers/"
        "paraphrase-multilingual-MiniLM-L12-v2"
    ),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Conecta ao banco ou cria a coleção, se necessário.
vectorstore_pacientes = Chroma(
    collection_name="prontuarios_sinteticos_v1",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_PATH)
)

documentos = []
ids = []

for paciente in pacientes:
    pid = paciente["patient_id"]

    # Documento 1: cadastro.
    cadastro = Document(
        page_content=(
            f"Prontuário: {pid}\n"
            f"Nome: {paciente['nome']}\n"
            f"Idade: {paciente['idade']}\n"
            f"Setor: {paciente['setor']}\n"
            f"E-mail fictício: {paciente['email']}\n"
            f"Telefone fictício: {paciente['telefone']}"
        ),
        metadata={
            "patient_id": pid,
            "record_type": "cadastro"
        }
    )

    # Documento 2: observações clínicas.
    notas_texto = "\n".join(
        f"{nota['id']}: {nota['texto']}"
        for nota in paciente["observacoes"]
    )

    notas = Document(
        page_content=(
            f"Prontuário: {pid}\n"
            f"Observações clínicas:\n{notas_texto}"
        ),
        metadata={
            "patient_id": pid,
            "record_type": "notas"
        }
    )

    # Documento 3: exames.
    exames_texto = "\n".join(
        f"{exame['id']}: {exame['nome']} "
        f"| Status: {exame['status']}"
        for exame in paciente["exames"]
    )

    exames = Document(
        page_content=(
            f"Prontuário: {pid}\n"
            f"Exames:\n{exames_texto}"
        ),
        metadata={
            "patient_id": pid,
            "record_type": "exames"
        }
    )

    documentos.extend([cadastro, notas, exames])

    ids.extend([
        f"cadastro::{pid}",
        f"notas::{pid}",
        f"exames::{pid}"
    ])

# Evita recalcular embeddings de registros já indexados.
ids_existentes = set(
    vectorstore_pacientes.get(include=[])["ids"]
)

pendentes = [
    (documento, doc_id)
    for documento, doc_id in zip(documentos, ids)
    if doc_id not in ids_existentes
]

print("Documentos esperados:", len(documentos))
print("Documentos para indexar:", len(pendentes))

# Processa em lotes para limitar o uso de memória.
tamanho_lote = 100

for inicio in range(0, len(pendentes), tamanho_lote):
    lote = pendentes[inicio:inicio + tamanho_lote]

    vectorstore_pacientes.add_documents(
        documents=[item[0] for item in lote],
        ids=[item[1] for item in lote]
    )

    print(
        f"Indexados: "
        f"{min(inicio + tamanho_lote, len(pendentes))}"
        f"/{len(pendentes)}"
    )

# Confirma a quantidade armazenada.
total_indexado = vectorstore_pacientes._collection.count()

assert total_indexado == 3000, (
    f"Esperados 3000 documentos, encontrados {total_indexado}"
)

print("\n[OK] Indexação concluída!")
print("Total de documentos:", total_indexado)
print("Banco persistido em:", CHROMA_PATH)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Documentos esperados: 3000
Documentos para indexar: 0

[OK] Indexação concluída!
Total de documentos: 3000
Banco persistido em: /content/drive/MyDrive/TechChallenge3/data/chroma_pacientes_v1


## 47. Recuperação de prontuários com controle de acesso

Implementamos uma função de recuperação dos registros armazenados
no ChromaDB, utilizando o identificador do paciente.

Antes de acessar os documentos, verificamos se o profissional
solicitante possui autorização.

Para a demonstração acadêmica, utilizamos permissões fictícias.

A consulta retorna os documentos de cadastro, observações
clínicas e exames, preservando suas respectivas fontes.

Este mecanismo não representa autenticação real de profissionais.

In [4]:
import re

# Permissões fictícias para os testes acadêmicos.
permissoes_demo = {
    ("clinico_demo_01", "PAC-DEMO-001"),
    ("clinico_demo_01", "PAC-DEMO-003"),
    ("clinico_demo_01", "PAC-DEMO-1000"),
    ("clinico_demo_02", "PAC-DEMO-002")
}


def recuperar_paciente(ator, patient_id):
    """
    Recupera os registros do paciente no ChromaDB
    somente após verificar a autorização.
    """

    # 1. Valida o identificador recebido.
    if not re.fullmatch(r"PAC-DEMO-\d{3,4}", patient_id):
        raise ValueError("Identificador de paciente inválido.")

    # 2. Verifica autorização ANTES da consulta.
    if (ator, patient_id) not in permissoes_demo:
        raise PermissionError(
            "Profissional não autorizado a acessar este paciente."
        )

    # 3. Busca os três documentos do paciente.
    ids = [
        f"cadastro::{patient_id}",
        f"notas::{patient_id}",
        f"exames::{patient_id}"
    ]

    resultado = vectorstore_pacientes.get(
        ids=ids,
        include=["documents", "metadatas"]
    )

    if len(resultado["ids"]) != 3:
        raise ValueError(
            "Prontuário incompleto ou não encontrado no ChromaDB."
        )

    # 4. Organiza os registros por tipo.
    registros = {}
    fontes = []

    for doc_id, texto, metadata in zip(
        resultado["ids"],
        resultado["documents"],
        resultado["metadatas"]
    ):
        # Verifica se o documento pertence ao paciente.
        if metadata["patient_id"] != patient_id:
            raise ValueError("Inconsistência de identificação.")

        tipo = metadata["record_type"]
        registros[tipo] = texto

        fontes.append({
            "documento": doc_id,
            "tipo": tipo,
            "paciente": patient_id
        })

    if set(registros) != {"cadastro", "notas", "exames"}:
        raise ValueError("Tipos de documentos inconsistentes.")

    return {
        "patient_id": patient_id,
        "registros": registros,
        "fontes": fontes
    }


# TESTE 1 — Acesso autorizado.
paciente_001 = recuperar_paciente(
    ator="clinico_demo_01",
    patient_id="PAC-DEMO-001"
)

assert "Marina Almeida Costa" in paciente_001["registros"]["cadastro"]
assert "PENDENTE" in paciente_001["registros"]["exames"]

print("[OK] Acesso autorizado.")
print("\nEXAMES:")
print(paciente_001["registros"]["exames"])
print("\nFONTES:")
print(paciente_001["fontes"])


# TESTE 2 — Acesso não autorizado.
try:
    recuperar_paciente(
        ator="clinico_demo_01",
        patient_id="PAC-DEMO-002"
    )
except PermissionError:
    print("\n[OK] Acesso indevido bloqueado.")
else:
    raise AssertionError("Falha no controle de acesso.")


# TESTE 3 — Último paciente.
paciente_1000 = recuperar_paciente(
    ator="clinico_demo_01",
    patient_id="PAC-DEMO-1000"
)

assert len(paciente_1000["fontes"]) == 3

print("[OK] PAC-DEMO-1000 recuperado.")
print("\n[OK] Testes de recuperação concluídos.")

[OK] Acesso autorizado.

EXAMES:
Prontuário: PAC-DEMO-001
Exames:
EX-0001-01: Hemograma | Status: PENDENTE
EX-0001-02: Glicemia | Status: CONCLUÍDO

FONTES:
[{'documento': 'cadastro::PAC-DEMO-001', 'tipo': 'cadastro', 'paciente': 'PAC-DEMO-001'}, {'documento': 'notas::PAC-DEMO-001', 'tipo': 'notas', 'paciente': 'PAC-DEMO-001'}, {'documento': 'exames::PAC-DEMO-001', 'tipo': 'exames', 'paciente': 'PAC-DEMO-001'}]

[OK] Acesso indevido bloqueado.
[OK] PAC-DEMO-1000 recuperado.

[OK] Testes de recuperação concluídos.


## 48. Agente RAG com Qwen

Nesta etapa, integramos o modelo Qwen2.5-0.5B-Instruct
à recuperação dos prontuários armazenados no ChromaDB.

O agente verifica a autorização antes de acessar os registros.

Informações estruturadas, como exames, são apresentadas
diretamente a partir do banco vetorial.

Para perguntas sobre observações clínicas, o Qwen recebe
somente o contexto recuperado do paciente autorizado.

As respostas geradas não representam validação clínica.

In [11]:
import torch
import unicodedata
from transformers import AutoTokenizer, AutoModelForCausalLM

# Modelo original utilizado exclusivamente pelo RAG.
MODELO_RAG = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer_rag = AutoTokenizer.from_pretrained(MODELO_RAG)

modelo_rag = AutoModelForCausalLM.from_pretrained(
    MODELO_RAG,
    dtype=torch.float32
).to("cpu")

modelo_rag.eval()

print("[OK] Qwen original carregado para o RAG.")


def gerar_texto_rag(mensagens, max_new_tokens=110):
    """
    Gera uma resposta com o Qwen a partir das mensagens
    e do contexto recuperado.
    """

    # Aplica o template de conversação do Qwen.
    prompt = tokenizer_rag.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True
    )

    # Converte o prompt em tensores PyTorch.
    entradas = tokenizer_rag(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to("cpu")

    with torch.inference_mode():
        saidas = modelo_rag.generate(
            **entradas,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer_rag.eos_token_id
        )

    # Remove os tokens pertencentes ao prompt.
    tamanho_entrada = entradas["input_ids"].shape[1]

    resposta = tokenizer_rag.decode(
        saidas[0][tamanho_entrada:],
        skip_special_tokens=True
    )

    return resposta.strip()


def responder_rag(pergunta, ator, patient_id, resposta_ft=None):
    """
    Responde perguntas sobre um paciente autorizado.
    Recupera dados no ChromaDB antes de gerar respostas.
    Um rascunho do FT pode apoiar a redação, mas nunca é fonte clínica.
    """

    # 1. Autoriza o acesso antes de qualquer consulta.
    dados = recuperar_paciente(
        ator=ator,
        patient_id=patient_id
    )

    pergunta_normalizada = pergunta.lower()

    # 2. Recuperação exata para exames.
    if any(
        termo in pergunta_normalizada
        for termo in ["exame", "hemograma", "glicemia"]
    ):
        texto_exames = dados["registros"]["exames"]

        if "pendente" in pergunta_normalizada:
            linhas = [
                linha
                for linha in texto_exames.splitlines()
                if "Status: PENDENTE" in linha
            ]

            resposta = (
                "\n".join(linhas)
                if linhas
                else "Nenhum exame pendente registrado."
            )
        else:
            resposta = texto_exames

        return {
            "resposta": resposta,
            "status": "recuperacao_exata",
            "fontes": [f"exames::{patient_id}"]
        }

    # 3. Busca semântica das anotações do paciente.
    documentos = vectorstore_pacientes.similarity_search(
        query=pergunta,
        k=1,
        filter={
            "$and": [
                {"patient_id": patient_id},
                {"record_type": "notas"}
            ]
        }
    )

    if not documentos:
        return {
            "resposta": "Não foram encontradas anotações clínicas.",
            "status": "sem_contexto",
            "fontes": []
        }

    contexto = "\n\n".join(
        documento.page_content
        for documento in documentos
    )

    # Consultas de sintomas/observações usam EXTRATOS LITERAIS das notas.
    # Evita que a LLM invente ou deforme termos clínicos (ex.: cefaleia).
    # O caminho FT -> RAG permanece; o rascunho FT não substitui a fonte.
    pergunta_sem_acentos = unicodedata.normalize(
        "NFKD", pergunta_normalizada
    )
    pergunta_sem_acentos = "".join(
        c for c in pergunta_sem_acentos if not unicodedata.combining(c)
    )
    consulta_de_notas = any(
        termo in pergunta_sem_acentos
        for termo in ("sintoma", "queixa", "observac", "anotac", "relato clinico")
    )
    if consulta_de_notas:
        trechos = []
        for documento in documentos:
            for linha in documento.page_content.splitlines():
                # A base sintética identifica cada observação como NOTA-0001.
                if re.match(r"^\s*NOTA-\d+\s*:", linha):
                    trecho = linha.split(":", 1)[1].strip()
                    if trecho and trecho not in trechos:
                        trechos.append(trecho)
        if trechos:
            return {
                "resposta": "Observações clínicas registradas (transcrição):\n" +
                           "\n".join(f"- {trecho}" for trecho in trechos),
                "status": "recuperacao_textual",
                "fontes": [f"notas::{patient_id}"]
            }
        # Não há registro extraível: não inventamos uma resposta narrativa.
        return {
            "resposta": "Não foi possível extrair uma observação clínica da fonte recuperada.",
            "status": "sem_contexto",
            "fontes": [f"notas::{patient_id}"]
        }

    # Quando o LangGraph percorre FT -> RAG, o texto do FT serve apenas
    # como sugestão de redação. Ele pode conter erros e NÃO é uma fonte.
    apoio_ft = (
        "\n\nRASCUNHO DO MODELO AJUSTADO (NÃO É FONTE CLÍNICA):\n"
        + resposta_ft[:800]
        + "\nIgnore qualquer afirmação sem apoio explícito nas notas acima."
        if resposta_ft else ""
    )

    mensagens = [
        {
            "role": "system",
            "content": (
                "Você é um assistente de consulta a prontuários "
                "sintéticos. Responda em português, de forma breve, "
                "utilizando exclusivamente as informações do contexto. "
                "Não invente sintomas, exames ou resultados. "
                "Não faça diagnósticos nem prescrições. "
                "Se não houver informação suficiente, diga isso. "
                "Se houver um rascunho de outro modelo, trate-o como "
                "não confiável e descarte fatos não presentes nas notas."
            )
        },
        {
            "role": "user",
            "content": (
                f"CONTEXTO:\n{contexto}\n\n"
                f"PERGUNTA:\n{pergunta}{apoio_ft}"
            )
        }
    ]

    resposta = gerar_texto_rag(mensagens)

    return {
        "resposta": resposta,
        "status": "geracao_nao_validada",
        "fontes": [
            f"notas::{patient_id}"
        ]
    }


# =========================================================
# TESTE 1 — Recuperação de exames
# =========================================================

print("\nTESTE 1 — EXAMES PENDENTES")

r1 = responder_rag(
    pergunta="Quais exames estão pendentes?",
    ator="clinico_demo_01",
    patient_id="PAC-DEMO-001"
)

print("Resposta:", r1["resposta"])
print("Fontes:", r1["fontes"])
print("Status:", r1["status"])

assert "Hemograma" in r1["resposta"]
assert "Glicemia" not in r1["resposta"]


# =========================================================
# TESTE 2 — Interpretação das observações clínicas
# =========================================================

print("\nTESTE 2 — SINTOMAS REGISTRADOS")

r2 = responder_rag(
    pergunta="Quais sintomas estão registrados?",
    ator="clinico_demo_01",
    patient_id="PAC-DEMO-001"
)

print("Resposta:", r2["resposta"])
print("Fontes:", r2["fontes"])
print("Status:", r2["status"])
assert r2["status"] == "recuperacao_textual"
assert "cefaleia" in r2["resposta"].lower()
assert "tontura" in r2["resposta"].lower()


# =========================================================
# TESTE 3 — Bloqueio de acesso indevido
# =========================================================

print("\nTESTE 3 — ACESSO NÃO AUTORIZADO")

try:
    responder_rag(
        pergunta="Quais sintomas estão registrados?",
        ator="clinico_demo_01",
        patient_id="PAC-DEMO-002"
    )
except PermissionError:
    print("[OK] Acesso indevido bloqueado.")
else:
    raise AssertionError("Falha no controle de acesso.")


print("\n[OK] Testes da célula 48 finalizados.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[OK] Qwen original carregado para o RAG.

TESTE 1 — EXAMES PENDENTES
Resposta: EX-0001-01: Hemograma | Status: PENDENTE
Fontes: ['exames::PAC-DEMO-001']
Status: recuperacao_exata

TESTE 2 — SINTOMAS REGISTRADOS
Resposta: Observações clínicas registradas (transcrição):
- Paciente relata episódios de cefaleia e tontura.
Fontes: ['notas::PAC-DEMO-001']
Status: recuperacao_textual

TESTE 3 — ACESSO NÃO AUTORIZADO
[OK] Acesso indevido bloqueado.

[OK] Testes da célula 48 finalizados.


## 49. Localização do modelo com Fine-Tuning

Verificamos a disponibilidade dos pesos do Qwen ajustado
anteriormente com o MedQuAD.

O objetivo é reutilizar o modelo treinado, sem executar
novamente o processo de Fine-Tuning.

Esta etapa não modifica arquivos e não carrega o modelo
na memória.

In [ ]:
from pathlib import Path

NOME_MODELO_FT = "qwen-medquad-full-ft-v2"

# Locais possíveis do modelo já extraído.
diretorios = [
    Path("/content/models") / NOME_MODELO_FT,
    BASE_PATH / "models" / NOME_MODELO_FT,
    BASE_PATH / NOME_MODELO_FT
]

# Inclui o caminho usado anteriormente, se disponível.
if "MODEL_DIR" in globals():
    diretorios.insert(0, Path(MODEL_DIR))

# Locais possíveis do ZIP salvo.
arquivos_zip = [
    Path("/content") / f"{NOME_MODELO_FT}.zip",
    BASE_PATH / "models" / f"{NOME_MODELO_FT}.zip",
    BASE_PATH / f"{NOME_MODELO_FT}.zip",
    Path("/content/drive/MyDrive") / f"{NOME_MODELO_FT}.zip"
]

modelo_encontrado = None
zip_encontrado = None

print("Verificando modelo com Fine-Tuning...\n")

for pasta in diretorios:
    if (
        pasta.is_dir()
        and (pasta / "config.json").exists()
        and list(pasta.glob("*.safetensors"))
    ):
        modelo_encontrado = pasta
        print("[OK] Modelo extraído:", pasta)
        break

for arquivo in arquivos_zip:
    if arquivo.is_file():
        zip_encontrado = arquivo
        print("[OK] ZIP encontrado:", arquivo)
        break

if modelo_encontrado is None and zip_encontrado is None:
    print("[ATENÇÃO] Modelo não encontrado nos caminhos verificados.")
    print("O ZIP pode estar somente no computador.")

print("\nVerificação finalizada.")

Verificando modelo com Fine-Tuning...

[ATENÇÃO] Modelo não encontrado nos caminhos verificados.
O ZIP pode estar somente no computador.

Verificação finalizada.


## 50. Orquestração dos agentes com LangGraph

Nesta etapa, implementamos o roteamento entre a consulta de prontuários,
o agente de perguntas médicas gerais e o bloqueio de solicitações indevidas.

As consultas de prontuários utilizam o agente RAG validado anteriormente.
A rota do Fine-Tuning permanece identificada como pendente até
restaurarmos o modelo ajustado.

Implementamos guardrails demonstrativos e um registro de auditoria
contendo o caminho executado, sem armazenar perguntas ou dados pessoais.

As regras de segurança não substituem controles hospitalares reais.

In [12]:
import json
import re
import unicodedata
from uuid import uuid4
from datetime import datetime, timezone
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


class EstadoAssistente(TypedDict, total=False):
    pergunta: str
    ator: str
    request_id: str
    patient_id: str
    rota: str
    resposta: str
    resposta_ft: str  # Rascunho experimental do FT, nunca fonte clínica.
    status: str
    fontes: list
    trilha: list


def normalizar_texto(texto):
    """Remove acentos e converte o texto para minúsculas."""
    texto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in texto if not unicodedata.combining(c))


def verificar_entrada(state):
    """Identifica paciente e autorização ANTES de qualquer acesso a dados."""
    pergunta = normalizar_texto(state["pergunta"])
    trilha = state.get("trilha", []) + ["verificar_entrada"]

    # Bloqueios demonstrativos permanecem ANTES dos agentes.
    if re.search(
        r"\b(prescreva|receite|diagnostique|posologia|dosagem|qual a dose)\b",
        pergunta
    ):
        return {
            "rota": "bloqueio", "status": "solicitacao_bloqueada",
            "resposta": "Não realizo prescrições, definição de doses ou diagnósticos autônomos.",
            "trilha": trilha
        }

    match_id = re.search(r"\bPAC-DEMO-\d{3,4}\b", state["pergunta"], re.I)
    patient_id = match_id.group(0).upper() if match_id else None

    if patient_id is None:
        encontrados = []
        for paciente in pacientes:
            nome = normalizar_texto(paciente["nome"])
            if re.search(r"(?<!\w)" + re.escape(nome) + r"(?!\w)", pergunta):
                encontrados.append(paciente)

        if encontrados:
            match_idade = re.search(r"\b(\d{1,3})\s*anos?\b", pergunta)
            if match_idade:
                idade = int(match_idade.group(1))
                encontrados = [p for p in encontrados if p["idade"] == idade]

            if len(encontrados) > 1:
                return {
                    "rota": "bloqueio", "status": "nome_ambiguo",
                    "resposta": "Existem múltiplos registros com esse nome. Informe também a idade ou o ID do prontuário.",
                    "trilha": trilha
                }
            if len(encontrados) == 1:
                patient_id = encontrados[0]["patient_id"]
            else:
                return {
                    "rota": "bloqueio", "status": "paciente_nao_identificado",
                    "resposta": "Não foi possível identificar o paciente com os dados informados.",
                    "trilha": trilha
                }

    # Nunca envia uma consulta identificada e não autorizada para o modelo.
    if patient_id is not None:
        if (state["ator"], patient_id) not in permissoes_demo:
            return {
                "rota": "bloqueio", "status": "acesso_negado",
                "resposta": "Acesso ao prontuário não autorizado.",
                "trilha": trilha
            }

        # Campos estruturados têm recuperação exata, sem precisar gerar texto.
        consulta_exata = any(termo in pergunta for termo in (
            "exame", "hemograma", "glicemia"
        ))
        return {
            "rota": "rag" if consulta_exata else "fine_tuning_rag",
            "patient_id": patient_id,
            "trilha": trilha
        }

    # Proíbe consultas a prontuário sem identificação confirmada.
    if any(termo in pergunta for termo in (
        "paciente", "prontuario", "exames pendentes", "exames do", "sintomas registrados"
    )):
        return {
            "rota": "bloqueio", "status": "paciente_nao_identificado",
            "resposta": "Não foi possível identificar o paciente. Informe dados suficientes para diferenciá-lo.",
            "trilha": trilha
        }

    # Perguntas médicas gerais entram no FT. O nó RAG seguinte não procura
    # prontuários se não houver paciente identificado/autorizado.
    return {"rota": "fine_tuning", "trilha": trilha}


def escolher_rota(state):
    return state["rota"]


def agente_rag(state):
    """RAG direto ou RAG após FT; sem patient_id não consulta prontuários."""
    trilha = state["trilha"] + ["agente_rag"]
    patient_id = state.get("patient_id")

    if not patient_id:
        # Pergunta geral: nó de passagem, SEM recuperação artificial de dados.
        return {
            "resposta": state.get("resposta_ft") or state.get("resposta", ""),
            "status": state.get("status", "sem_resposta"),
            "fontes": [], "trilha": trilha
        }

    resultado = responder_rag(
        pergunta=state["pergunta"],
        ator=state["ator"],
        patient_id=patient_id,
        resposta_ft=state.get("resposta_ft")
    )
    return {
        "resposta": resultado["resposta"], "status": resultado["status"],
        "fontes": resultado["fontes"], "trilha": trilha
    }


def agente_fine_tuning(state):
    """Implementação provisória, substituída pela célula 54."""
    return {
        "resposta_ft": "",
        "resposta": "O modelo com Fine-Tuning ainda não foi restaurado neste ambiente.",
        "status": "fine_tuning_pendente", "fontes": [],
        "trilha": state["trilha"] + ["agente_fine_tuning"]
    }


def agente_bloqueio(state):
    return {"fontes": [], "trilha": state["trilha"] + ["agente_bloqueio"]}


def validar_saida(state):
    """Segurança pós-agentes; não substitui validação clínica."""
    resposta = (state.get("resposta") or "").strip()
    status = state.get("status", "sem_resposta")
    fontes = state.get("fontes", [])

    if not resposta:
        resposta = "Não foi possível produzir uma resposta; revisão humana necessária."
        status = "sem_resposta"

    # Bloqueio demonstrativo de instruções explícitas de dosagem na saída.
    if status in ("geracao_nao_validada", "recuperacao_exata", "recuperacao_textual") and re.search(
        r"\b(?:tome|administre|aplique|ingerir)\s+\d+(?:[.,]\d+)?\s*(?:mg|ml|comprimidos?)\b",
        normalizar_texto(resposta), re.I
    ):
        resposta = "Resposta bloqueada por possível orientação medicamentosa. Solicite revisão profissional."
        status = "resposta_bloqueada"
        fontes = []
    elif status == "geracao_nao_validada":
        resposta += "\n\n[Aviso: conteúdo gerado por IA, sem validação clínica. Revisão profissional necessária.]"

    if state.get("rota") == "bloqueio":
        fontes = []

    return {
        "resposta": resposta, "status": status, "fontes": fontes,
        "trilha": state["trilha"] + ["validar_saida"]
    }


def registrar_auditoria(state):
    """Registra somente a rota e o percurso, sem perguntas nem prontuários."""
    trilha = state["trilha"] + ["registrar_auditoria"]
    registro = {
        "request_id": state["request_id"],
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "rota": state["rota"], "status": state["status"],
        "caminho": trilha,
        "quantidade_fontes": len(state.get("fontes", []))
    }
    arquivo = RESULTS_PATH / "auditoria_langgraph.jsonl"
    with open(arquivo, "a", encoding="utf-8") as f:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")
    return {"trilha": trilha}


def montar_grafo():
    """Input -> FT -> RAG -> Security ou Input -> RAG -> Security."""
    workflow = StateGraph(EstadoAssistente)
    workflow.add_node("verificar_entrada", verificar_entrada)
    workflow.add_node("agente_rag", agente_rag)
    workflow.add_node("agente_fine_tuning", agente_fine_tuning)
    workflow.add_node("agente_bloqueio", agente_bloqueio)
    workflow.add_node("validar_saida", validar_saida)
    workflow.add_node("registrar_auditoria", registrar_auditoria)

    workflow.add_edge(START, "verificar_entrada")
    workflow.add_conditional_edges(
        "verificar_entrada", escolher_rota,
        {
            "rag": "agente_rag",
            "fine_tuning_rag": "agente_fine_tuning",
            "fine_tuning": "agente_fine_tuning",
            "bloqueio": "agente_bloqueio"
        }
    )
    workflow.add_edge("agente_fine_tuning", "agente_rag")
    workflow.add_edge("agente_rag", "validar_saida")
    workflow.add_edge("agente_bloqueio", "validar_saida")
    workflow.add_edge("validar_saida", "registrar_auditoria")
    workflow.add_edge("registrar_auditoria", END)
    return workflow.compile()


grafo = montar_grafo()


def consultar_assistente(pergunta, ator="clinico_demo_01"):
    """Executa pergunta pelo LangGraph completo."""
    return grafo.invoke({
        "pergunta": pergunta, "ator": ator,
        "request_id": str(uuid4()), "trilha": []
    })


print("[OK] Grafo preparado: RAG direto ou Fine-Tuning -> RAG, segurança e auditoria.")
print("[INFO] O nó Fine-Tuning real será conectado na célula 54.")


[OK] Grafo preparado: RAG direto ou Fine-Tuning -> RAG, segurança e auditoria.
[INFO] O nó Fine-Tuning real será conectado na célula 54.


## 51. Validação integrada do assistente médico

Nesta etapa, verificamos o funcionamento do LangGraph,
executando perguntas representativas de cada rota.

Os testes avaliam:

- Recuperação de exames do paciente autorizado.
- Geração de respostas baseadas em anotações clínicas.
- Bloqueio de acesso não autorizado.
- Bloqueio de solicitações diretas de prescrição.
- Encaminhamento de perguntas gerais para a rota Fine-Tuning.

Também verificamos os registros de auditoria e os caminhos
efetivamente percorridos.

A rota Fine-Tuning permanece pendente de carregamento do
modelo ajustado. Os testes são funcionais, não clínicos.

In [13]:

from pathlib import Path
import zipfile

# Pasta onde o modelo será restaurado no Colab.
PASTA_MODELOS = Path("/content/models")
PASTA_FT = PASTA_MODELOS / "qwen-medquad-full-ft-v2"

# Procura o ZIP nos locais conhecidos.
possiveis_zips = [
    BASE_PATH / "models" / "qwen-medquad-full-ft-v2.zip",
    BASE_PATH / "qwen-medquad-full-ft-v2.zip",
    Path("/content/qwen-medquad-full-ft-v2.zip")
]

zip_ft = next(
    (arquivo for arquivo in possiveis_zips if arquivo.is_file()),
    None
)

# Evita extrair novamente se o modelo já estiver completo.
modelo_restaurado = (
    (PASTA_FT / "config.json").is_file()
    and bool(list(PASTA_FT.glob("*.safetensors")))
)

if modelo_restaurado:
    print("[OK] Modelo já estava extraído.")

else:
    if zip_ft is None:
        raise FileNotFoundError(
            "ZIP não encontrado. Envie "
            "qwen-medquad-full-ft-v2.zip "
            "para a pasta TechChallenge3/models do Drive."
        )

    print("[OK] ZIP encontrado:", zip_ft)
    print("Restaurando os pesos do Fine-Tuning...")

    PASTA_MODELOS.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_ft, "r") as arquivo:
        nomes = arquivo.namelist()

        # Confirma a estrutura do ZIP que geramos
        # ao exportar o modelo treinado.
        assert (
            "qwen-medquad-full-ft-v2/config.json" in nomes
        ), "Estrutura do ZIP diferente da esperada."

        # Impede a extração de caminhos inseguros.
        for membro in arquivo.infolist():
            caminho = Path(membro.filename)

            if caminho.is_absolute() or ".." in caminho.parts:
                raise ValueError(
                    "Caminho inseguro encontrado no ZIP."
                )

        arquivo.extractall(PASTA_MODELOS)

# Verifica os arquivos essenciais.
assert (PASTA_FT / "config.json").is_file(), (
    "Configuração do modelo não encontrada."
)

assert list(PASTA_FT.glob("*.safetensors")), (
    "Pesos do Fine-Tuning não encontrados."
)

print("\n[OK] Modelo Fine-Tuning restaurado!")
print("Diretório:", PASTA_FT)
print("Arquivos disponíveis:")

for arquivo in sorted(PASTA_FT.iterdir()):
    if arquivo.is_file():
        print("-", arquivo.name)


[OK] Modelo já estava extraído.

[OK] Modelo Fine-Tuning restaurado!
Diretório: /content/models/qwen-medquad-full-ft-v2
Arquivos disponíveis:
- chat_template.jinja
- config.json
- generation_config.json
- model.safetensors
- tokenizer.json
- tokenizer_config.json
- training_args.bin
- training_metrics.json


## 52. Restauração do modelo ajustado com Fine-Tuning

Nesta etapa, restauramos os pesos do Qwen2.5-0.5B-Instruct
treinado anteriormente com o dataset MedQuAD.

O treinamento completo foi realizado em aproximadamente
1h02 e seus pesos foram exportados para um arquivo ZIP.

A restauração não realiza novo treinamento nem modifica
os pesos do modelo.

O modelo será carregado para inferência na próxima etapa.

In [7]:

from pathlib import Path
import zipfile

# Pasta onde o modelo será restaurado no Colab.
PASTA_MODELOS = Path("/content/models")
PASTA_FT = PASTA_MODELOS / "qwen-medquad-full-ft-v2"

# Procura o ZIP nos locais conhecidos.
possiveis_zips = [
    BASE_PATH / "models" / "qwen-medquad-full-ft-v2.zip",
    BASE_PATH / "qwen-medquad-full-ft-v2.zip",
    Path("/content/qwen-medquad-full-ft-v2.zip")
]

zip_ft = next(
    (arquivo for arquivo in possiveis_zips if arquivo.is_file()),
    None
)

# Evita extrair novamente se o modelo já estiver completo.
modelo_restaurado = (
    (PASTA_FT / "config.json").is_file()
    and bool(list(PASTA_FT.glob("*.safetensors")))
)

if modelo_restaurado:
    print("[OK] Modelo já estava extraído.")

else:
    if zip_ft is None:
        raise FileNotFoundError(
            "ZIP não encontrado. Envie "
            "qwen-medquad-full-ft-v2.zip "
            "para a pasta TechChallenge3/models do Drive."
        )

    print("[OK] ZIP encontrado:", zip_ft)
    print("Restaurando os pesos do Fine-Tuning...")

    PASTA_MODELOS.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_ft, "r") as arquivo:
        nomes = arquivo.namelist()

        # Confirma a estrutura do ZIP que geramos
        # ao exportar o modelo treinado.
        assert (
            "qwen-medquad-full-ft-v2/config.json" in nomes
        ), "Estrutura do ZIP diferente da esperada."

        # Impede a extração de caminhos inseguros.
        for membro in arquivo.infolist():
            caminho = Path(membro.filename)

            if caminho.is_absolute() or ".." in caminho.parts:
                raise ValueError(
                    "Caminho inseguro encontrado no ZIP."
                )

        arquivo.extractall(PASTA_MODELOS)

# Verifica os arquivos essenciais.
assert (PASTA_FT / "config.json").is_file(), (
    "Configuração do modelo não encontrada."
)

assert list(PASTA_FT.glob("*.safetensors")), (
    "Pesos do Fine-Tuning não encontrados."
)

print("\n[OK] Modelo Fine-Tuning restaurado!")
print("Diretório:", PASTA_FT)
print("Arquivos disponíveis:")

for arquivo in sorted(PASTA_FT.iterdir()):
    if arquivo.is_file():
        print("-", arquivo.name)


[OK] ZIP encontrado: /content/drive/MyDrive/TechChallenge3/models/qwen-medquad-full-ft-v2.zip
Restaurando os pesos do Fine-Tuning...

[OK] Modelo Fine-Tuning restaurado!
Diretório: /content/models/qwen-medquad-full-ft-v2
Arquivos disponíveis:
- chat_template.jinja
- config.json
- generation_config.json
- model.safetensors
- tokenizer.json
- tokenizer_config.json
- training_args.bin
- training_metrics.json


## 53. Carregamento e teste do modelo com Fine-Tuning

Nesta etapa, carregamos o Qwen2.5-0.5B-Instruct ajustado
com o dataset MedQuAD.

Os pesos foram recuperados do treinamento Full Fine-Tuning
realizado anteriormente, sem necessidade de novo treinamento.

Realizamos uma inferência com uma pergunta médica geral
para verificar o funcionamento do modelo ajustado.

A resposta será analisada posteriormente, pois uma geração
bem-sucedida não comprova precisão ou segurança clínica.

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Diretório restaurado na célula 52.
MODELO_FT_PATH = PASTA_FT

# 1. Carrega o tokenizer do nosso modelo ajustado.
tokenizer_ft = AutoTokenizer.from_pretrained(
    str(MODELO_FT_PATH)
)

# 2. Carrega os pesos resultantes do Full Fine-Tuning.
# Utilizamos CPU para não depender da disponibilidade de GPU.
modelo_ft = AutoModelForCausalLM.from_pretrained(
    str(MODELO_FT_PATH),
    dtype=torch.float32,
    low_cpu_mem_usage=True
).to("cpu")

# Modo de inferência: não atualiza os pesos.
modelo_ft.eval()

print("[OK] Modelo Fine-Tuning carregado.")
print("Dispositivo:", next(modelo_ft.parameters()).device)


def gerar_resposta_ft(pergunta, max_new_tokens=120):
    """
    Gera uma resposta usando exclusivamente
    o Qwen ajustado com MedQuAD.
    """

    mensagens = [
        {
            "role": "system",
            "content": (
                "You are a medical information assistant. "
                "Provide a concise, factual answer. "
                "Do not invent information or prescribe treatments. "
                "If uncertain, acknowledge uncertainty."
            )
        },
        {
            "role": "user",
            "content": pergunta
        }
    ]

    # Aplica o template original de conversação do Qwen.
    prompt = tokenizer_ft.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokeniza a entrada corretamente.
    entradas = tokenizer_ft(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to("cpu")

    # Geração determinística, sem gradientes.
    with torch.inference_mode():
        saidas = modelo_ft.generate(
            **entradas,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer_ft.eos_token_id
        )

    # Remove os tokens correspondentes à pergunta.
    tamanho_prompt = entradas["input_ids"].shape[1]

    resposta = tokenizer_ft.decode(
        saidas[0][tamanho_prompt:],
        skip_special_tokens=True
    )

    return resposta.strip()


# =====================================================
# TESTE — Pergunta médica geral
# =====================================================

pergunta_teste = "What is diabetes mellitus?"

print("\nPERGUNTA:", pergunta_teste)

resposta_teste = gerar_resposta_ft(pergunta_teste)

print("\nRESPOSTA DO MODELO AJUSTADO:")
print(resposta_teste)

print("\n[OK] Teste de inferência finalizado.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[OK] Modelo Fine-Tuning carregado.
Dispositivo: cpu

PERGUNTA: What is diabetes mellitus?

RESPOSTA DO MODELO AJUSTADO:
Diabetes mellitus (DM) is a chronic disease that occurs when the body does not make enough insulin or cannot use insulin effectively. Insulin is a hormone made by the pancreas, an organ located behind the stomach. The pancreas releases insulin into the bloodstream to help control blood sugar levels. When people with DM do not have enough insulin, their blood glucose levels rise above normal and can cause serious health problems such as heart attack, stroke, vision loss, kidney failure, amputation of feet and legs, and death. People with DM may also develop other complications including high blood pressure, nerve

[OK] Teste de inferência finalizado.


## 54. Integração do modelo Fine-Tuning ao LangGraph

Nesta etapa, conectamos o Qwen ajustado com MedQuAD
ao nó responsável por perguntas médicas gerais.

O agente RAG continua responsável exclusivamente pelas
informações dos prontuários sintéticos.

O LangGraph utiliza o modelo ajustado na rota Fine-Tuning,
sem consultar o ChromaDB.

As respostas geradas são identificadas como não validadas
clinicamente e exigem revisão profissional.

A integração reutiliza os pesos treinados anteriormente,
sem realizar novo treinamento.

In [9]:
# =========================================================
# 54. CONEXÃO DOS PESOS AJUSTADOS AO LANGGRAPH
# =========================================================

def agente_fine_tuning(state):
    """FT prepara resposta geral/rascunho; RAG verifica dados do paciente."""
    pergunta = state["pergunta"]
    if state.get("patient_id"):
        # Não envia ao FT os registros recuperados nem dados do ChromaDB.
        # O FT fornece apenas um rascunho geral, NÃO fatos do prontuário.
        nome = next(
            (p["nome"] for p in pacientes if p["patient_id"] == state["patient_id"]),
            ""
        )
        if nome:
            pergunta = re.sub(re.escape(nome), "um paciente fictício", pergunta, flags=re.I)

    rascunho = gerar_resposta_ft(
        pergunta=pergunta,
        max_new_tokens=80 if state.get("patient_id") else 160
    )
    status = "geracao_nao_validada" if rascunho else "sem_resposta"
    resposta = rascunho or "O modelo não conseguiu gerar uma resposta."
    # Em consultas com paciente, o nó RAG vai substituir esta resposta
    # pelo texto limitado ao contexto realmente recuperado.
    return {
        "resposta_ft": rascunho,
        "resposta": resposta,
        "status": status,
        "fontes": [],
        "trilha": state["trilha"] + ["agente_fine_tuning"]
    }


# A função montar_grafo() da célula 50 preserva as mesmas arestas;
# recompilamos somente para conectar o nó Fine-Tuning real.
grafo = montar_grafo()
print("[OK] Grafo recompilado com modelo FT real -> RAG -> segurança.")


# TESTE: pergunta médica geral. O nó RAG é atravessado SEM consulta
# a prontuários, porque não existe paciente identificado.
resultado_ft = consultar_assistente("What is diabetes mellitus?")
print("Rota:", resultado_ft["rota"])
print("Status:", resultado_ft["status"])
print("Caminho:", " -> ".join(resultado_ft["trilha"]))
print("Resposta:", resultado_ft["resposta"])
print("Fontes:", resultado_ft["fontes"])

assert resultado_ft["rota"] == "fine_tuning"
assert resultado_ft["status"] in ("geracao_nao_validada", "resposta_bloqueada")
assert resultado_ft["trilha"].index("agente_fine_tuning") < resultado_ft["trilha"].index("agente_rag")
assert resultado_ft["fontes"] == []
assert resultado_ft["trilha"][-2:] == ["validar_saida", "registrar_auditoria"]
print("[OK] Qwen ajustado integrado; nenhum prontuário foi consultado.")


[OK] Grafo recompilado com modelo FT real -> RAG -> segurança.
Rota: fine_tuning
Status: geracao_nao_validada
Caminho: verificar_entrada -> agente_fine_tuning -> agente_rag -> validar_saida -> registrar_auditoria
Resposta: Diabetes mellitus (DM) is a chronic disease that occurs when the body does not make enough insulin or cannot use insulin effectively. Insulin is a hormone made by the pancreas, an organ located behind the stomach. The pancreas releases insulin into the bloodstream to help control blood sugar levels. When people with DM do not have enough insulin, their blood glucose levels rise above normal and can cause serious health problems such as heart attack, stroke, vision loss, kidney failure, amputation of feet and legs, and death. People with DM may also develop other complications including high blood pressure, nerve damage in the hands and feet, and skin problems. Diabetes mellitus is one type of metabolic syndrome. Metabolic syndrome is a group of conditions that occur 

## 55. Validação final do assistente médico

Esta etapa valida a integração dos agentes implementados com LangGraph.

### Cenários avaliados

1. Consulta de exames pelo identificador do paciente.
2. Consulta de exames pelo nome completo do paciente.
3. Tratamento de nomes duplicados.
4. Tratamento de pacientes não identificados.
5. Bloqueio de acesso não autorizado.
6. Bloqueio de solicitações de prescrição.
7. Verificação da execução do modelo com Fine-Tuning.
8. Verificação dos registros de auditoria.

O sistema identifica o paciente pelo nome, resolve seu identificador
interno e verifica a autorização antes da consulta ao ChromaDB.

Os testes verificam o comportamento funcional do protótipo.
Não constituem validação clínica ou de segurança para produção.

In [14]:

# =========================================================
# 55. VALIDAÇÃO FINAL DO ASSISTENTE MÉDICO
# =========================================================

import json
from collections import Counter


def executar_teste(nome, pergunta, rota, status):
    """
    Executa um teste pelo LangGraph e verifica
    o roteamento, o status e a auditoria.
    """
    resultado = consultar_assistente(pergunta)

    assert resultado["rota"] == rota, (
        f"Rota incorreta no teste: {nome}"
    )
    assert resultado["status"] == status, (
        f"Status incorreto no teste: {nome}"
    )
    assert resultado["trilha"][-2:] == ["validar_saida", "registrar_auditoria"]

    print(f"\n{'=' * 55}")
    print("TESTE:", nome)
    print("Pergunta:", pergunta)
    print("Rota:", resultado["rota"])
    print("Status:", resultado["status"])
    print("Caminho:", " -> ".join(resultado["trilha"]))
    print("Resposta:", resultado["resposta"])
    print("Fontes:", resultado.get("fontes", []))

    return resultado


# =========================================================
# TESTE 1 — CONSULTA PELO IDENTIFICADOR
# =========================================================

teste_id = executar_teste(
    nome="Consulta pelo ID",
    pergunta="Quais exames estão pendentes para PAC-DEMO-001?",
    rota="rag",
    status="recuperacao_exata"
)

assert "Hemograma" in teste_id["resposta"]
assert "Glicemia" not in teste_id["resposta"]


# =========================================================
# TESTE 2 — CONSULTA PELO NOME COMPLETO
# =========================================================

paciente_demo = next(
    p for p in pacientes
    if p["patient_id"] == "PAC-DEMO-001"
)

pergunta_nome = (
    f"Quais exames estão pendentes para a paciente "
    f"{paciente_demo['nome']}, "
    f"{paciente_demo['idade']} anos?"
)

teste_nome = executar_teste(
    nome="Consulta pelo nome",
    pergunta=pergunta_nome,
    rota="rag",
    status="recuperacao_exata"
)

assert "Hemograma" in teste_nome["resposta"]
assert "agente_fine_tuning" not in teste_nome["trilha"]

print("[OK] Nome convertido para prontuário autorizado.")


# =========================================================
# TESTE 2B — RESPOSTA NARRATIVA: FINE-TUNING -> RAG
# =========================================================

# O mesmo paciente autorizado é identificado por nome e idade.
# O FT gera um rascunho geral; o RAG consulta as notas e fundamenta
# a resposta final nos dados efetivamente recuperados.
teste_combinado = executar_teste(
    nome="Sintomas: Fine-Tuning seguido de RAG",
    pergunta=(
        f"Quais sintomas estão registrados para a paciente "
        f"{paciente_demo['nome']}, {paciente_demo['idade']} anos?"
    ),
    rota="fine_tuning_rag",
    status="recuperacao_textual"
)

# A resposta textual deve reproduzir os termos da nota original,
# não a paráfrase incorreta observada no teste anterior.
assert "cefaleia" in teste_combinado["resposta"].lower()
assert "tontura" in teste_combinado["resposta"].lower()
assert "cefalorraquina" not in teste_combinado["resposta"].lower()
assert "Paciente relata episódios de cefaleia e tontura." in teste_combinado["resposta"]

assert teste_combinado["trilha"].index("agente_fine_tuning") < (
    teste_combinado["trilha"].index("agente_rag")
)
assert teste_combinado["fontes"] == ["notas::PAC-DEMO-001"]
assert "agente_bloqueio" not in teste_combinado["trilha"]


# =========================================================
# TESTE 3 — NOMES DUPLICADOS
# =========================================================

contagem_nomes = Counter(
    p["nome"] for p in pacientes
)

nomes_duplicados = [
    nome for nome, quantidade in contagem_nomes.items()
    if quantidade > 1
]

if nomes_duplicados:
    nome_duplicado = nomes_duplicados[0]

    teste_duplicado = executar_teste(
        nome="Nome duplicado",
        pergunta=(
            f"Quais exames estão pendentes para "
            f"{nome_duplicado}?"
        ),
        rota="bloqueio",
        status="nome_ambiguo"
    )

    assert "agente_rag" not in teste_duplicado["trilha"]
else:
    print("\n[INFO] Nenhum nome duplicado na base.")


# =========================================================
# TESTE 4 — PACIENTE NÃO IDENTIFICADO
# =========================================================

teste_ausente = executar_teste(
    nome="Paciente não identificado",
    pergunta="Quais exames estão pendentes para o paciente Eric?",
    rota="bloqueio",
    status="paciente_nao_identificado"
)

assert "agente_rag" not in teste_ausente["trilha"]


# =========================================================
# TESTE 5 — ACESSO NÃO AUTORIZADO
# =========================================================

teste_acesso = executar_teste(
    nome="Acesso não autorizado",
    pergunta="Quais exames existem para PAC-DEMO-002?",
    rota="bloqueio",
    status="acesso_negado"
)

assert "agente_rag" not in teste_acesso["trilha"]
assert teste_acesso.get("fontes", []) == []


# =========================================================
# TESTE 6 — SOLICITAÇÃO DE PRESCRIÇÃO
# =========================================================

teste_prescricao = executar_teste(
    nome="Solicitação de prescrição",
    pergunta="Prescreva um medicamento para PAC-DEMO-001",
    rota="bloqueio",
    status="solicitacao_bloqueada"
)

assert "agente_fine_tuning" not in teste_prescricao["trilha"]
assert "agente_rag" not in teste_prescricao["trilha"]


# =========================================================
# TESTE 7 — MODELO COM FINE-TUNING
# =========================================================

# Reutiliza o teste real realizado na célula 54,
# evitando gerar novamente uma resposta em CPU.

assert resultado_ft["rota"] == "fine_tuning"
assert resultado_ft["status"] in ("geracao_nao_validada", "resposta_bloqueada")
assert "agente_fine_tuning" in resultado_ft["trilha"]
assert "agente_rag" in resultado_ft["trilha"]  # Sem recuperação: fontes vazias

print(f"\n{'=' * 55}")
print("TESTE: Fine-Tuning integrado")
print("Rota:", resultado_ft["rota"])
print("Status:", resultado_ft["status"])
print("Caminho:", " -> ".join(resultado_ft["trilha"]))
print("[OK] Modelo ajustado executado na célula 54.")


# =========================================================
# TESTE 8 — AUDITORIA
# =========================================================

arquivo_auditoria = (
    RESULTS_PATH / "auditoria_langgraph.jsonl"
)

assert arquivo_auditoria.exists()

with open(arquivo_auditoria, "r", encoding="utf-8") as f:
    registros = [
        json.loads(linha)
        for linha in f
        if linha.strip()
    ]

# Confirma os registros desta execução,
# incluindo o teste de Fine-Tuning da célula 54.

resultados = [
    teste_id,
    teste_nome,
    teste_combinado,
    teste_ausente,
    teste_acesso,
    teste_prescricao,
    resultado_ft
]

if nomes_duplicados:
    resultados.append(teste_duplicado)

ids_testados = {
    resultado["request_id"]
    for resultado in resultados
}

registros_testados = {
    registro["request_id"]: registro
    for registro in registros
    if registro["request_id"] in ids_testados
}

assert set(registros_testados) == ids_testados

for registro in registros_testados.values():
    assert registro["caminho"][-2:] == ["validar_saida", "registrar_auditoria"]

    # O log não deve registrar o texto das perguntas
    # nem das respostas clínicas.
    assert "pergunta" not in registro
    assert "resposta" not in registro

print(f"\n{'=' * 55}")
print("[OK] Auditoria confirmada.")
print("Registros desta validação:", len(registros_testados))
print("Arquivo:", arquivo_auditoria)


# =========================================================
# RESULTADO FINAL
# =========================================================

print("\n" + "=" * 55)
print("[OK] VALIDAÇÃO FUNCIONAL CONCLUÍDA!")
print("=" * 55)

print("[OK] Consultas por ID e nome completo (RAG direto)")
print("[OK] Consulta narrativa: Fine-Tuning -> RAG -> segurança")
print("[OK] Tratamento de nomes duplicados")
print("[OK] Identificação de pacientes ausentes")
print("[OK] Controle de acesso demonstrativo")
print("[OK] Bloqueio de prescrição direta")
print("[OK] Fine-Tuning integrado")
print("[OK] Auditoria dos caminhos executados")



TESTE: Consulta pelo ID
Pergunta: Quais exames estão pendentes para PAC-DEMO-001?
Rota: rag
Status: recuperacao_exata
Caminho: verificar_entrada -> agente_rag -> validar_saida -> registrar_auditoria
Resposta: EX-0001-01: Hemograma | Status: PENDENTE
Fontes: ['exames::PAC-DEMO-001']

TESTE: Consulta pelo nome
Pergunta: Quais exames estão pendentes para a paciente Marina Almeida Costa, 53 anos?
Rota: rag
Status: recuperacao_exata
Caminho: verificar_entrada -> agente_rag -> validar_saida -> registrar_auditoria
Resposta: EX-0001-01: Hemograma | Status: PENDENTE
Fontes: ['exames::PAC-DEMO-001']
[OK] Nome convertido para prontuário autorizado.

TESTE: Sintomas: Fine-Tuning seguido de RAG
Pergunta: Quais sintomas estão registrados para a paciente Marina Almeida Costa, 53 anos?
Rota: fine_tuning_rag
Status: recuperacao_textual
Caminho: verificar_entrada -> agente_fine_tuning -> agente_rag -> validar_saida -> registrar_auditoria
Resposta: Observações clínicas registradas (transcrição):
- Pacie

## 56. Teste realizando o input no terminal

In [36]:
from uuid import uuid4

pergunta = input("Digite sua pergunta: ").strip()

resultado = grafo.invoke({
    "pergunta": pergunta,
    "ator": "clinico_demo_01",
    "request_id": str(uuid4()),
    "trilha": []
})

print("\n" + "=" * 55)
print("ASSISTENTE MÉDICO — RESPOSTA")
print("=" * 55)

print(f"\n{resultado.get('resposta', 'Sem resposta disponível.')}")

print("\n" + "-" * 55)
print("DETALHES DA EXECUÇÃO")
print("-" * 55)

print(f"Rota: {resultado.get('rota', 'Não informada')}")
print(f"Status: {resultado.get('status', 'Não informado')}")

trilha = resultado.get("trilha", [])
print(f"Fluxo: {' → '.join(trilha)}")

fontes = resultado.get("fontes", [])
if fontes:
    print("\nFontes consultadas:")
    for fonte in fontes:
        print(f"  - {fonte}")

print(f"\nID da requisição: {resultado.get('request_id', '-')}")
print("=" * 55)

Digite sua pergunta: Quais sintomas estão registrados para Rafael Santos Silva?

ASSISTENTE MÉDICO — RESPOSTA

Rafael Santos Silva apresenta desconforto abdominal intermitente.

[Aviso: resposta gerada automaticamente e sujeita à revisão profissional.]

-------------------------------------------------------
DETALHES DA EXECUÇÃO
-------------------------------------------------------
Rota: rag
Status: geracao_nao_validada
Fluxo: verificar_entrada → agente_rag → registrar_auditoria

Fontes consultadas:
  - notas::PAC-DEMO-006

ID da requisição: 0c99fb8e-88d5-4538-b82e-56100cb3a0e9
